In [17]:
import os
from pathlib import Path

def find_image_captions(image_name, csv_files):
    """
    Cherche une image dans plusieurs fichiers CSV et retourne les captions trouvés
    
    Args:
        image_name: nom de l'image à chercher
        csv_files: liste des fichiers CSV à parcourir
    
    Returns:
        dict: {nom_fichier: caption} pour chaque fichier contenant l'image
    """
    results = {}
    
    for csv_file in csv_files:
        with open(csv_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line.startswith(image_name):
                    # Extraire le caption après le délimiteur |
                    parts = line.split('|', 1)
                    if len(parts) == 2:
                        caption = parts[1].strip()
                        # Utiliser le nom du fichier comme clé
                        file_name = Path(csv_file).stem  # a1, a2, etc.
                        results[file_name] = caption
                        break  # Passer au fichier suivant
    
    return results

if __name__ == "__main__":
    # Chemin vers tes fichiers CSV
    csv_dir = ""
    
    # Charger tous les fichiers CSV (a1.csv, a2.csv, etc.)
    csv_files = sorted(Path(csv_dir).glob("*.csv"))
    
    print(f"✓ {len(csv_files)} fichiers CSV trouvés\n")
    
    # Exemple : chercher une image
    image_name = "1115679311_245eff2f4b.jpg"
    captions_dict = find_image_captions(image_name, csv_files)
    
    print(f"Image recherchée: {image_name}")
    print(f"Résultats: {captions_dict}\n")
    
    # Affichage formaté
    if captions_dict:
        print("Captions trouvés:")
        for file_name, caption in captions_dict.items():
            print(f"  {file_name}: {caption}")
    else:
        print("❌ Aucun caption trouvé pour cette image")

✓ 6 fichiers CSV trouvés

Image recherchée: 1115679311_245eff2f4b.jpg
Résultats: {'generated_captions_cnn_lstm': 'weather batting dot seashore chest site site site agility enclosure descending animal dot intersection cheeks splits dot intersection cheeks crocodile dot gym beginning peak drain splits hooping stump piano stove kicked jogs claus peak cheek wheeled upset gym beginning round firing wheeled lacrosse portrait struggle bouncy electric intersection pit saying', 'generated_captions_cnn_transformer': 'hound puppy fetches a handgun into home plate of war with onlookers nearby', 'generated_captions_ft_vgg_dense_lstm': 'a man in a red shirt and blue jeans is standing on a sidewalk', 'generated_captions_ft_vgg_dense_transformer': 'a black dog is walking on the beach .', 'generated_captions_vgg_dense_lstm': 'a man and a dog are in the water', 'generated_captions_vlm': 'a man is lying on the beach.'}

Captions trouvés:
  generated_captions_cnn_lstm: weather batting dot seashore chest s

In [19]:
"""
Script d'évaluation BLEU-1 et BERTScore pour Image Captioning
"""

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score
import numpy as np
import nltk

# Télécharger les ressources NLTK
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)


# ============================================================================
# DONNÉES
# ============================================================================

image = "1115679311_245eff2f4b.jpg"

generated_captions = {
    "generated_captions_cnn_lstm": "weather batting dot seashore chest site site site agility enclosure descending animal dot intersection cheeks splits dot intersection cheeks crocodile dot gym beginning peak drain splits hooping stump piano stove kicked jogs claus peak cheek wheeled upset gym beginning round firing wheeled lacrosse portrait struggle bouncy electric intersection pit saying",
    "generated_captions_cnn_transformer": "hound puppy fetches a handgun into home plate of war with onlookers nearby",
    "generated_captions_ft_vgg_dense_lstm": "a man in a red shirt and blue jeans is standing on a sidewalk",
    "generated_captions_ft_vgg_dense_transformer": "a black dog is walking on the beach .",
    "generated_captions_vgg_dense_lstm": "a man and a dog are in the water",
    "generated_captions_vlm": "a man is lying on the beach.",
    "generated_captions_vgg_dense_transformer": "a black dog is running on the beach ."
}

real_captions = [
    "A black dog is jumping over a log along a beach .",
    "A black dog runs along the beach and jumps over a wood barrior .",
    "A dog is running on the beach .",
    "a dog jumping over a small wall at a beach near the water .",
    "An animal is running on the beach ."
]


# ============================================================================
# FONCTIONS D'ÉVALUATION
# ============================================================================

def calculate_bleu1(generated, references):
    """Calcule BLEU-1"""
    gen_tokens = generated.lower().split()
    ref_tokens = [ref.lower().split() for ref in references]
    
    smoothing = SmoothingFunction()
    score = sentence_bleu(
        ref_tokens, 
        gen_tokens, 
        weights=(1, 0, 0, 0),
        smoothing_function=smoothing.method1
    )
    return score


def calculate_bertscore(generated, references):
    """Calcule BERTScore moyen sur toutes les références"""
    scores = []
    
    for ref in references:
        P, R, F1 = bert_score(
            [generated], 
            [ref], 
            lang='en',
            verbose=False,
            model_type='bert-base-uncased'
        )
        scores.append({
            'P': P.item(),
            'R': R.item(),
            'F1': F1.item()
        })
    
    avg_P = np.mean([s['P'] for s in scores])
    avg_R = np.mean([s['R'] for s in scores])
    avg_F1 = np.mean([s['F1'] for s in scores])
    
    return avg_P, avg_R, avg_F1


# ============================================================================
# ÉVALUATION
# ============================================================================

print("=" * 80)
print(f"ÉVALUATION - Image: {image}")
print("=" * 80)

results = {}

for model_name, caption in generated_captions.items():
    print(f"\n{'='*80}")
    print(f"🤖 Modèle: {model_name}")
    print(f"{'='*80}")
    print(f"Caption générée: {caption}")
    print()
    
    # BLEU-1
    bleu1 = calculate_bleu1(caption, real_captions)
    print(f"📊 BLEU-1:  {bleu1:.4f}")
    
    # BERTScore
    print(f"\n⏳ Calcul BERTScore...")
    bert_P, bert_R, bert_F1 = calculate_bertscore(caption, real_captions)
    
    print(f"📊 BERTScore (moyenne sur {len(real_captions)} références):")
    print(f"   - Precision: {bert_P:.4f}")
    print(f"   - Recall:    {bert_R:.4f}")
    print(f"   - F1:        {bert_F1:.4f}")
    
    results[model_name] = {
        'BLEU-1': bleu1,
        'BERTScore_P': bert_P,
        'BERTScore_R': bert_R,
        'BERTScore_F1': bert_F1
    }


# ============================================================================
# RÉSUMÉ
# ============================================================================

print("\n\n" + "=" * 80)
print("📋 RÉSUMÉ DES RÉSULTATS")
print("=" * 80)

print("\n{:<40} {:>12} {:>12}".format("Modèle", "BLEU-1", "BERTScore F1"))
print("-" * 80)
for model, scores in results.items():
    model_short = model.replace("generated_captions_", "")
    print("{:<40} {:>12.4f} {:>12.4f}".format(
        model_short, 
        scores['BLEU-1'], 
        scores['BERTScore_F1']
    ))

# Meilleurs modèles
print("\n" + "=" * 80)
print("🏆 MEILLEURS MODÈLES")
print("=" * 80)

best_bleu = max(results.items(), key=lambda x: x[1]['BLEU-1'])
best_bert = max(results.items(), key=lambda x: x[1]['BERTScore_F1'])

print(f"\n🥇 Meilleur BLEU-1:")
print(f"   {best_bleu[0].replace('generated_captions_', '')}")
print(f"   Score: {best_bleu[1]['BLEU-1']:.4f}")

print(f"\n🥇 Meilleur BERTScore F1:")
print(f"   {best_bert[0].replace('generated_captions_', '')}")
print(f"   Score: {best_bert[1]['BERTScore_F1']:.4f}")

print("\n" + "=" * 80)

ModuleNotFoundError: No module named 'bert_score'

In [ ]:
image = "1115679311_245eff2f4b.jpg"

generated_captions = {
    "generated_captions_cnn_lstm": "weather batting dot seashore chest site site site agility enclosure descending animal dot intersection cheeks splits dot intersection cheeks crocodile dot gym beginning peak drain splits hooping stump piano stove kicked jogs claus peak cheek wheeled upset gym beginning round firing wheeled lacrosse portrait struggle bouncy electric intersection pit saying",
    "generated_captions_cnn_transformer": "hound puppy fetches a handgun into home plate of war with onlookers nearby",
    "generated_captions_ft_vgg_dense_lstm": "a man in a red shirt and blue jeans is standing on a sidewalk",
    "generated_captions_ft_vgg_dense_transformer": "a black dog is walking on the beach .",
    "generated_captions_vgg_dense_lstm": "a man and a dog are in the water",
    "generated_captions_vlm": "a man is lying on the beach.",
    "generated_captions_vgg_dense_transformer": "a black dog is running on the beach ."
}

# UNE SEULE référence
real_caption = "A black dog is jumping over a log along a beach ."

Modèle                                         BLEU-1 BERTScore F1
--------------------------------------------------------------------------------
vgg_dense_transformer                          0.4296       0.8382
ft_vgg_dense_transformer                       0.4296       0.8332
vlm                                            0.2420       0.7183
vgg_dense_lstm                                 0.2669       0.6169
ft_vgg_dense_lstm                              0.2857       0.5599
cnn_transformer                                0.0769       0.4830
cnn_lstm                                       0.0000       0.3721

In [ ]:
image = "1075867198_27ca2e7efe.jpg"

generated_captions={
"generated_captions_cnn_lstm": "splits hooping guitars arena cheek obstacle tattoo chased rails pit formation airport flute exercise windsurfer rowboat saber end push angels upset trunks tiger perched crossed framed butter upset cone pigtails rowboat beak rowboat beak suit pulled rails wheeled itself pigtails recently striking beginning beginning beginning vertical weather legged squatting wheeled",
"generated_captions_cnn_transformer": "clad cowgirl gear parasails on leashes whilst bearing its mouth open hawaiian",
"generated_captions_ft_vgg_dense_lstm": "a man in a red shirt and blue jeans is standing on a sidewalk",
"generated_captions_ft_vgg_dense_transformer": "two dogs are playing on the beach .",
"generated_captions_vgg_dense_lstm": "a man and a dog are standing in the water",
"generated_captions_vlm": "a man wearing a blue shirt and a white shirt and a black shirt and white shirt and white shirt and white shirt",
"generated_captions_vgg_dense_transformer": "a man and a dog are playing with a ball in the sand ."
}
real_captions = ["A man in a brown shirt and dark shorts plays on the beach with his two black dogs ."]


Modèle                                         BLEU-1 BERTScore F1
--------------------------------------------------------------------------------
ft_vgg_dense_transformer                       0.0807       0.7682
vgg_dense_transformer                          0.3077       0.7144
vgg_dense_lstm                                 0.2715       0.6390
ft_vgg_dense_lstm                              0.2857       0.5599
cnn_transformer                                0.0000       0.4615
vlm                                            0.1739       0.4272
cnn_lstm                                       0.0000       0.3752

In [ ]:
image = "103195344_5d2dc613a3.jpg"

generated_captions={
"generated_captions_cnn_lstm": "asking puck site agility angels recently 8 products incline pumpkin framed just rescue thumbs mats instrument framed cream peak chest sidelines footballers bow raised tied bottoms porch reflective slopes dot finish driven rusty bales bales mats trumpet hurdles hoop watched motion wheeled lacrosse kangaroo pigtails skiiers tattoo chased rails beginning",
"generated_captions_cnn_transformer": "clad cowgirl gear parasails on leashes whilst bearing its mouth open doorway",
"generated_captions_ft_vgg_dense_lstm": "a man in a red shirt and blue jeans is standing on a sidewalk",
"generated_captions_ft_vgg_dense_transformer": "a boy is standing on a sidewalk with a large building in the background .",
"generated_captions_vgg_dense_lstm": "a man in a black shirt and jeans is jumping over a pole",
"generated_captions_vlm": "a man is sitting on a wooden staircase.",
"generated_captions_vgg_dense_transformer": "a boy is sitting on a sidewalk with a large black and white building in the background ."
}
real_captions = ["A man in black is sitting next to a modern art structure in front of a glass building ."]


Modèle                                         BLEU-1 BERTScore F1
--------------------------------------------------------------------------------
vlm                                            0.2577       0.6960
vgg_dense_lstm                                 0.5385       0.6353
ft_vgg_dense_transformer                       0.2857       0.6347
vgg_dense_transformer                          0.2941       0.6140
ft_vgg_dense_lstm                              0.2857       0.5599
cnn_transformer                                0.0000       0.4599
cnn_lstm                                       0.0000       0.3714

In [ ]:
image = "1095980313_3c94799968.jpg"

generated_captions = {
    "generated_captions_cnn_lstm": "bottles 23 agility angels angels framed products incline beginning descending cone emerging hilly tinkerbell stump link splits dot bay gymnastics reddish that clapping shephard products incline pumpkin framed just eyed plates incline batting pigtails marker located enclosure trumpet just rescue cats tattoo chased squirted steering lion medieval kangaroo ribbon framed",
    "generated_captions_cnn_transformer": "clad cowgirl gear parasails on leashes whilst bearing its mouth open hawaiian",
    "generated_captions_ft_vgg_dense_lstm": "a man in a red shirt and blue jeans is standing on a sidewalk",
    "generated_captions_ft_vgg_dense_transformer": "a young girl in a pink shirt is standing on a ledge beside a beach .",
    "generated_captions_vgg_dense_lstm": "a boy in a blue shirt and blue shorts is jumping into a pool",
    "generated_captions_vlm": "a young woman is jumping on the beach.",
    "generated_captions_vgg_dense_transformer": "a young girl is standing on a ledge overlooking a clear blue sky ."
}

# UNE SEULE référence
real_caption = "A little girl in a blue swimsuit is walking along a wooden fence next to a sandy beach ."


Modèle                                         BLEU-1 BERTScore F1
--------------------------------------------------------------------------------
ft_vgg_dense_transformer                       0.4367       0.8185
vgg_dense_transformer                          0.3142       0.6966
vgg_dense_lstm                                 0.3221       0.6770
vlm                                            0.1074       0.6704
ft_vgg_dense_lstm                              0.3221       0.6641
cnn_transformer                                0.0000       0.4413
cnn_lstm                                       0.0000       0.3118